<a href="https://colab.research.google.com/github/kimmy111-zhu/human-validation/blob/main/notebooks/alpacafarm_matched_sampling-typo%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import random
import os
import pandas as pd
from google.colab import files


# =========================================================
# 1. Settings
# =========================================================

RANDOM_SEED = 42

# 这里的 5 指 5 个共同匹配成功的原始 prompt
# 最终会输出 5 × 3 = 15 行
SAMPLE_SIZE = 5

BENCHMARK_NAME = "AlpacaFarm"

TYPO_RATES = ["0.1", "0.4", "0.7"]


# =========================================================
# 2. Automatically locate files
# =========================================================

def find_file(filename):
    """
    自动检查文件是在 /content/ 下，
    还是在 /content/alpaca_farm/ 文件夹中。
    """

    possible_paths = [
        f"/content/{filename}",
        f"/content/alpaca_farm/{filename}"
    ]

    for file_path in possible_paths:
        if os.path.exists(file_path):
            return file_path

    raise FileNotFoundError(
        f"Could not find {filename}.\n"
        f"Please upload it to /content/ or "
        f"/content/alpaca_farm/."
    )


FILE_PATHS = {
    "0.1": find_file("raw_0.1.jsonl"),
    "0.4": find_file("raw_0.4.jsonl"),
    "0.7": find_file("raw_0.7.jsonl")
}

print("Files found:")

for typo_rate, file_path in FILE_PATHS.items():
    print(f"Rate {typo_rate}: {file_path}")


# =========================================================
# 3. Helper functions
# =========================================================

def read_jsonl(file_path):
    """
    Read a JSONL file and preserve its original row number.
    """

    records = []

    with open(file_path, "r", encoding="utf-8-sig") as file:
        for line_number, line in enumerate(file, start=1):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON on line {line_number} "
                    f"in {file_path}: {error}"
                )

            record["_source_row"] = line_number
            records.append(record)

    return records


def normalize_text(value):
    """
    Normalize text only for matching.

    This removes unnecessary differences caused by
    spaces and line breaks.
    """

    if value is None:
        return ""

    if isinstance(value, (dict, list)):
        value = json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True
        )

    return " ".join(str(value).split())


def get_first_available(record, possible_fields):
    """
    Return the first non-empty field found in a record.
    """

    for field_name in possible_fields:
        value = record.get(field_name)

        if value is not None and str(value).strip() != "":
            return value

    return ""


def get_original_prompt(record):
    """
    Extract the original prompt used for matching.
    """

    return get_first_available(
        record,
        [
            "original_text_backup",
            "original_prompt",
            "original_instruction",
            "original_text",
            "prompt_original"
        ]
    )


def get_modified_prompt(record):
    """
    Extract the modified AlpacaFarm instruction/prompt.
    """

    return get_first_available(
        record,
        [
            "instruction",
            "question",
            "prompt",
            "modified_prompt",
            "text",
            "input"
        ]
    )


def get_gold_answer(record):
    """
    Extract the reference answer, if one exists.
    """

    return get_first_available(
        record,
        [
            "output",
            "answer",
            "reference_answer",
            "gold_answer",
            "response",
            "reference"
        ]
    )


# =========================================================
# 4. Read all three files
# =========================================================

datasets = {}

for typo_rate, file_path in FILE_PATHS.items():

    datasets[typo_rate] = read_jsonl(file_path)

    print(
        f"Loaded {len(datasets[typo_rate])} records "
        f"for typo rate {typo_rate}"
    )


# Display available fields for checking
print("\nFields found in the first record of each file:")

for typo_rate in TYPO_RATES:

    if datasets[typo_rate]:
        print(
            f"\nRate {typo_rate}: "
            f"{list(datasets[typo_rate][0].keys())}"
        )


# =========================================================
# 5. Create matching maps
# =========================================================

record_maps = {}

for typo_rate, records in datasets.items():

    current_map = {}
    missing_original_count = 0
    duplicate_count = 0

    for record in records:

        original_prompt = get_original_prompt(record)
        normalized_original = normalize_text(original_prompt)

        if not normalized_original:
            missing_original_count += 1
            continue

        if normalized_original in current_map:
            duplicate_count += 1
            continue

        current_map[normalized_original] = record

    record_maps[typo_rate] = current_map

    print(
        f"\nRate {typo_rate}: "
        f"{len(current_map)} usable original prompts"
    )

    if missing_original_count > 0:
        print(
            f"Warning: {missing_original_count} records "
            f"had no original prompt."
        )

    if duplicate_count > 0:
        print(
            f"Warning: {duplicate_count} duplicate original prompts "
            f"were found. The first record was kept."
        )


# =========================================================
# 6. Find prompts shared by all three typo-rate files
# =========================================================

common_original_keys = set(record_maps["0.1"].keys())

for typo_rate in ["0.4", "0.7"]:
    common_original_keys &= set(
        record_maps[typo_rate].keys()
    )

common_original_keys = sorted(common_original_keys)

print(
    f"\nCommon matched original prompts: "
    f"{len(common_original_keys)}"
)


if len(common_original_keys) < SAMPLE_SIZE:
    raise ValueError(
        f"Only {len(common_original_keys)} matched prompts "
        f"were found across all three files.\n"
        f"Cannot sample {SAMPLE_SIZE} prompts."
    )


# =========================================================
# 7. Randomly sample 5 matched original prompts
# =========================================================

random.seed(RANDOM_SEED)

selected_original_keys = random.sample(
    common_original_keys,
    SAMPLE_SIZE
)

print(f"\nRandom seed: {RANDOM_SEED}")
print(f"Selected matched prompts: {SAMPLE_SIZE}")


# =========================================================
# 8. Create matched and stratified output
# =========================================================

output_rows = []

for prompt_number, original_key in enumerate(
    selected_original_keys,
    start=1
):

    base_id = f"{BENCHMARK_NAME}_{prompt_number:03d}"

    # Use the original prompt stored in the 0.1 record
    reference_record = record_maps["0.1"][original_key]

    original_prompt = get_original_prompt(reference_record)

    for typo_rate in TYPO_RATES:

        record = record_maps[typo_rate][original_key]

        modified_prompt = get_modified_prompt(record)
        gold_answer = get_gold_answer(record)

        output_rows.append({
            "Base_ID": base_id,

            "Sample_ID": (
                f"{base_id}_rate_{typo_rate}"
            ),

            "Benchmark": BENCHMARK_NAME,
            "Typo_Rate": typo_rate,

            "Source_File": os.path.basename(
                FILE_PATHS[typo_rate]
            ),

            "Source_Row": record.get(
                "_source_row",
                ""
            ),

            "Original_Prompt": original_prompt,
            "Modified_Prompt": modified_prompt,
            "Gold_Answer": gold_answer,

            "R1_Meaning": "",
            "R2_Meaning": "",
            "Final_Meaning": "",

            "R1_Key_Info": "",
            "R2_Key_Info": "",
            "Final_Key_Info": "",

            "R1_Answer_Preserved": "",
            "R2_Answer_Preserved": "",
            "Final_Answer_Preserved": "",

            "R1_Realism": "",
            "R2_Realism": "",
            "Final_Realism": "",

            "R1_Readability": "",
            "R2_Readability": "",
            "Final_Readability": "",

            "R1_Comments": "",
            "R2_Comments": "",
            "Final_Comments": ""
        })


# =========================================================
# 9. Convert to DataFrame
# =========================================================

sample_df = pd.DataFrame(output_rows)

print("\nSampling completed.")
print(f"Selected original prompts: {SAMPLE_SIZE}")
print(f"Number of typo-rate strata: {len(TYPO_RATES)}")
print(f"Total output rows: {len(sample_df)}")

display(sample_df)


# =========================================================
# 10. Check whether important columns are empty
# =========================================================

empty_modified = (
    sample_df["Modified_Prompt"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_original = (
    sample_df["Original_Prompt"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_answer = (
    sample_df["Gold_Answer"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("\nColumn check:")
print(f"Empty Original_Prompt rows: {empty_original}")
print(f"Empty Modified_Prompt rows: {empty_modified}")
print(f"Empty Gold_Answer rows: {empty_answer}")

if empty_modified > 0:
    print(
        "\nWarning: Some Modified_Prompt values are empty.\n"
        "Please check the field names printed above. "
        "The modified AlpacaFarm prompt may use another field."
    )

if empty_answer > 0:
    print(
        "\nNote: Some Gold_Answer values are empty. "
        "This may be normal if the raw AlpacaFarm files "
        "do not contain a reference answer."
    )


# =========================================================
# 11. Save and download CSV
# =========================================================

output_file = (
    "/content/"
    "AlpacaFarm_matched_stratified_sample_"
    "n5_seed42.csv"
)

sample_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nCSV saved successfully: {output_file}")

files.download(output_file)

Files found:
Rate 0.1: /content/raw_0.1.jsonl
Rate 0.4: /content/raw_0.4.jsonl
Rate 0.7: /content/raw_0.7.jsonl
Loaded 805 records for typo rate 0.1
Loaded 805 records for typo rate 0.4
Loaded 805 records for typo rate 0.7

Fields found in the first record of each file:

Rate 0.1: ['datasplit', 'dataset', 'instruction', 'input', 'output', 'generator', 'sample_mode', 'typo_rate_metadata', 'original_text_backup', 'typo_source_field', '_source_row']

Rate 0.4: ['datasplit', 'dataset', 'instruction', 'input', 'output', 'generator', 'sample_mode', 'typo_rate_metadata', 'original_text_backup', 'typo_source_field', '_source_row']

Rate 0.7: ['datasplit', 'dataset', 'instruction', 'input', 'output', 'generator', 'sample_mode', 'typo_rate_metadata', 'original_text_backup', 'typo_source_field', '_source_row']

Rate 0.1: 804 usable original prompts

Rate 0.4: 804 usable original prompts

Rate 0.7: 804 usable original prompts

Common matched original prompts: 804

Random seed: 42
Selected matched 

,Base_ID,Sample_ID,Benchmark,Typo_Rate,Source_File,Source_Row,Original_Prompt,Modified_Prompt,Gold_Answer,R1_Meaning,...,Final_Answer_Preserved,R1_Realism,R2_Realism,Final_Realism,R1_Readability,R2_Readability,Final_Readability,R1_Comments,R2_Comments,Final_Comments
0,AlpacaFarm_001,AlpacaFarm_001_rate_0.1,AlpacaFarm,0.1,raw_0.1.jsonl,285,Write a detailed patent writing for an innovat...,Wrtie a detailed patent writing for an innovat...,This patent writing details an innovative and ...,,...,,,,,,,,,,
1,AlpacaFarm_001,AlpacaFarm_001_rate_0.4,AlpacaFarm,0.4,raw_0.4.jsonl,285,Write a detailed patent writing for an innovat...,Writre a detailed patent writng for am innovti...,This patent writing details an innovative and ...,,...,,,,,,,,,,
2,AlpacaFarm_001,AlpacaFarm_001_rate_0.7,AlpacaFarm,0.7,raw_0.7.jsonl,285,Write a detailed patent writing for an innovat...,Write a detaile patent writingh frt an innovat...,This patent writing details an innovative and ...,,...,,,,,,,,,,
3,AlpacaFarm_002,AlpacaFarm_002_rate_0.1,AlpacaFarm,0.1,raw_0.1.jsonl,566,Design a programming problem related to the su...,Design a programming problem related to the su...,Design a programming problem related to Dynami...,,...,,,,,,,,,,
4,AlpacaFarm_002,AlpacaFarm_002_rate_0.4,AlpacaFarm,0.4,raw_0.4.jsonl,566,Design a programming problem related to the su...,Design a programmign problem relatewd to the s...,Design a programming problem related to Dynami...,,...,,,,,,,,,,
5,AlpacaFarm_002,AlpacaFarm_002_rate_0.7,AlpacaFarm,0.7,raw_0.7.jsonl,566,Design a programming problem related to the su...,Dwesig a programming problen relatd to tjer su...,Design a programming problem related to Dynami...,,...,,,,,,,,,,
6,AlpacaFarm_003,AlpacaFarm_003_rate_0.1,AlpacaFarm,0.1,raw_0.1.jsonl,754,"As a space colonist on Mars, describe your dai...","As a space clonist on Mars, describe yoiur dai...","As a space colonist on Mars, my daily life is ...",,...,,,,,,,,,,
7,AlpacaFarm_003,AlpacaFarm_003_rate_0.4,AlpacaFarm,0.4,raw_0.4.jsonl,754,"As a space colonist on Mars, describe your dai...","A a sacr colonist on Mars, descrbie your daily...","As a space colonist on Mars, my daily life is ...",,...,,,,,,,,,,
8,AlpacaFarm_003,AlpacaFarm_003_rate_0.7,AlpacaFarm,0.7,raw_0.7.jsonl,754,"As a space colonist on Mars, describe your dai...","As a spce colonist ob Msrs, descrbie yourt dai...","As a space colonist on Mars, my daily life is ...",,...,,,,,,,,,,
9,AlpacaFarm_004,AlpacaFarm_004_rate_0.1,AlpacaFarm,0.1,raw_0.1.jsonl,212,is queue an ADT or a data structure,is queue an AD or a data structure,Queue is both an ADT (Abstract Data Type) and ...,,...,,,,,,,,,,



Column check:
Empty Original_Prompt rows: 0
Empty Modified_Prompt rows: 0
Empty Gold_Answer rows: 0

CSV saved successfully: /content/AlpacaFarm_matched_stratified_sample_n5_seed42.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>